# Codex Installer
> Install MCP configuration for OpenAI Codex

In [ ]:
#| default_exp install.codex

In [ ]:
#| hide
from nbdev.showdoc import *

## Imports

In [ ]:
#| export
from __future__ import annotations
from pathlib import Path
from typing import Optional

from nbdev_mcp.install.base import (
    BaseInstaller, InstallResult, MCPServerConfig,
    which, run_command, get_python_path,
    read_toml_config, write_toml_config,
    get_codex_config_path
)

## TOML Utilities

## Codex Installer

In [ ]:
#| export

class CodexInstaller(BaseInstaller):
    """Installer for OpenAI Codex.
    
    Supports two installation methods:
    1. Native CLI: `codex mcp add` (preferred)
    2. Config file: Write to ~/.codex/config.toml
    
    Parameters
    ----------
    use_cli : bool
        Prefer CLI installation if available.
    """
    
    def __init__(self, use_cli: bool = True):
        self.use_cli = use_cli
    
    @property
    def name(self) -> str:
        return 'codex'
    
    @property
    def config_path(self) -> Path:
        return get_codex_config_path()
    
    @property
    def has_cli(self) -> bool:
        """Check if Codex CLI is available."""
        return which('codex') is not None
    
    def is_configured(self) -> bool:
        """Check if nbdev is already configured."""
        config = read_toml_config(self.config_path)
        return 'nbdev' in config.get('mcp_servers', {})
    
    def install_via_cli(self, server: MCPServerConfig) -> bool:
        """Install using Codex CLI."""
        if not self.has_cli:
            return False
        
        # Build command: codex mcp add nbdev -- python -m nbdev_mcp
        args = [
            'codex', 'mcp', 'add',
            server.name,
            '--',
            server.command,
            *server.args
        ]
        
        result = run_command(args)
        return result.returncode == 0
    
    def uninstall_via_cli(self) -> bool:
        """Uninstall using Codex CLI."""
        if not self.has_cli:
            return False
        
        result = run_command(['codex', 'mcp', 'remove', 'nbdev'])
        return result.returncode == 0
    
    def do_install(self, server: MCPServerConfig) -> None:
        """Perform the installation."""
        # Try CLI first if enabled
        if self.use_cli and self.has_cli:
            if self.install_via_cli(server):
                return
        
        # Fall back to config file
        config = read_toml_config(self.config_path)
        if 'mcp_servers' not in config:
            config['mcp_servers'] = {}
        
        config['mcp_servers'][server.name] = {
            'command': server.command,
            'args': server.args
        }
        write_toml_config(self.config_path, config)
    
    def do_uninstall(self) -> None:
        """Remove the configuration."""
        # Try CLI first if enabled
        if self.use_cli and self.has_cli:
            if self.uninstall_via_cli():
                return
        
        # Fall back to config file
        config = read_toml_config(self.config_path)
        if 'mcp_servers' in config and 'nbdev' in config['mcp_servers']:
            del config['mcp_servers']['nbdev']
        write_toml_config(self.config_path, config)

## Convenience Functions

In [ ]:
#| export
def install_codex(
    python_path: Optional[str] = None,
    force: bool = False,
    dry_run: bool = False,
    use_cli: bool = True
) -> InstallResult:
    """Install nbdev-mcp configuration for Codex.
    
    Parameters
    ----------
    python_path : str, optional
        Python interpreter path. Defaults to current.
    force : bool
        Overwrite existing configuration.
    dry_run : bool
        Show what would be done without making changes.
    use_cli : bool
        Use `codex mcp add` CLI if available.
    
    Returns
    -------
    InstallResult
        Result of the installation.
    """
    installer = CodexInstaller(use_cli=use_cli)
    server = MCPServerConfig.for_nbdev(python_path)
    return installer.install(server, force=force, dry_run=dry_run)


def uninstall_codex(dry_run: bool = False, use_cli: bool = True) -> InstallResult:
    """Remove nbdev-mcp configuration from Codex."""
    installer = CodexInstaller(use_cli=use_cli)
    return installer.uninstall(dry_run=dry_run)

## Export

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()